## Self notes — running log

Short observations only. One markdown cell per note. Add a code cell underneath if a demo is genuinely useful — most of the time it isn't, the practice notebooks already cover that.


### 1 — `SparkSession` has two versions

- `pyspark.sql.session.SparkSession` — **classic**, drives a local JVM.
- `pyspark.sql.connect.session.SparkSession` — **Connect**, talks to a remote Spark server over gRPC.

Same API surface, different classes. Helpers that need to touch the JVM directly (e.g. `delta.configure_spark_with_delta_pip(builder)`) accept only the classic builder and reject the Connect one with a type-mismatch error.


### 2 — Predicate pushdown: what does and doesn't push

- **In-memory sources can never push filters.** `Range` (`spark.range`), `LocalTableScan` / `ExistingRDD` (`createDataFrame`) have no scan layer to push into — the data is already in memory. The `Filter` just sits above the source.
- **Even when a filter pushes, the `Filter` node stays in the plan.** Source-level pushdown is best-effort (e.g. Parquet uses row-group min/max stats — it can skip whole groups but rows inside surviving groups still need exact re-checking). Catalyst keeps the `Filter` above the scan for correctness.

```text
*(1) Project [customer_id#0, full_name#1, credit_score#3L]
  +- *(1) Filter (isnotnull(credit_score#3L) AND (credit_score#3L >= 700))   ← exact re-check
     +- *(1) ColumnarToRow
        +- FileScan parquet [...]
             PushedFilters: [IsNotNull(credit_score), GreaterThanOrEqual(credit_score,700)]
             ReadSchema: struct<customer_id:string,full_name:string,credit_score:bigint>
```


### 3 — `foreach(print)` vs `show()`

Both are actions, but they print from different places and behave very differently:

| | `df.show()` | `df.foreach(print)` |
|---|---|---|
| Where the print happens | Driver | Executors |
| Where you see the output | Your console | Executor stdout / cluster logs — usually invisible in a notebook or `spark-submit --master yarn` |
| Output format | Pretty ASCII table, header + 20 rows by default | One `Row(...)` per line, unordered, no header |
| Order | First 20 rows in source order (top-down) | Whatever order partitions emit — interleaved across executors |
| Data movement | First N rows pulled to driver | None — runs on each row in place |
| Safe at scale | Yes (bounded) | Yes (no driver memory pressure), but log volume can be huge |

In `local[*]` mode the executor *is* the driver, so `foreach(print)` happens to land in the same console — which is why it looked equivalent to `show()` in the practice script. On a real cluster the prints disappear into executor logs and you'd think nothing happened.

Rule of thumb: use `show()` for inspecting data; reserve `foreach` for genuine per-row side effects (writing to a sink, pushing to a queue) and even then prefer `foreachPartition` to amortize connection setup.


### 4 — Temp views vs. tables

**Temp views — four flavours along two axes:**

|  | Errors if it exists | Replaces silently |
|---|---|---|
| **Session-scoped** | `createTempView(name)` | `createOrReplaceTempView(name)` |
| **Cross-session (application-scoped)** | `createGlobalTempView(name)` | `createOrReplaceGlobalTempView(name)` |

- Session views are visible only inside the current `SparkSession`. Global views live in a special `global_temp` database visible to every `SparkSession` in the same Spark application — both die when the application stops.
- Reference syntax: session view → `SELECT * FROM v`. Global view → `SELECT * FROM global_temp.v` (the prefix is mandatory).
- All four are **metadata only** — they register a name pointing at the DataFrame's logical plan. No data is written. Every query re-executes the upstream transformations (unless the underlying DataFrame is cached).
- Non-`OrReplace` variants throw `AnalysisException` when a name collides — useful as a guardrail when re-registering would silently be a bug.

**Tables — persistent in a catalog:**

| Kind | Created with | Data ownership | Drop behaviour | Lifetime |
|---|---|---|---|---|
| **Managed table** | `df.write.saveAsTable("t")` or `CREATE TABLE t … USING <fmt>` (no `LOCATION`) | Spark / the metastore owns the files | `DROP TABLE` deletes both metadata **and** the underlying files | Survives application restarts |
| **External table** | `df.write.option("path", "/loc").saveAsTable("t")` or `CREATE TABLE t … LOCATION '/loc'` | You own the files | `DROP TABLE` deletes only the metadata; files stay | Survives application restarts |

- Tables live in a catalog (Hive metastore, AWS Glue, Unity Catalog, …). Anything that can connect to the catalog can see them — across sessions, jobs, even applications.
- Managed table files are written under `spark.sql.warehouse.dir` by default. External tables go wherever you point them.
- A table can be backed by any format: Parquet, ORC, JSON, Delta, etc. The `USING delta` clause is what makes it a Delta table (and unlocks ACID, time travel, etc.).

**Rule of thumb:** use `createOrReplaceTempView` for scratch in notebooks; reach for a real table when results need to outlive the session, be queried from elsewhere, or get scheduled refreshes.


### 5 — `spark.stop()` vs `sc.stop()` — one is enough, both is harmless

`SparkSession` wraps a single underlying `SparkContext`. `spark.stop()` calls `sc.stop()` internally, so one call shuts both down.

- `spark.stop()` alone → enough. Conventional choice (session is the higher-level handle).
- `sc.stop()` alone → also enough. The session has no life without its context.
- Both, in either order → also fine, the second call is a silent no-op.

**Why no error on the second call?** `stop()` is **idempotent**. The JVM `SparkContext.stop()` does `stopped.compareAndSet(false, true)` first — if it's already `true`, the rest of the method is skipped. Designed that way because `stop()` is the kind of thing that ends up in `finally:` blocks, `atexit` handlers, signal handlers, JVM shutdown hooks, and IDE stop buttons — having shutdown itself raise would mask the real exit reason.

**The real guard is on use, not on close.** Calling `stop()` twice is fine; calling an *operation* on a stopped context is not:

```python
sc.stop()
sc.parallelize([1, 2, 3])   # IllegalStateException: SparkContext has been shut down
```


### 6 — RDD creation/transform options, grouped by purpose

Per-method option names are inconsistent (`numSlices` vs `minPartitions` vs `numPartitions`). Forgettable individually, easy once grouped by **what the option does**.

**Bucket 1 — Partition count:**

| Option | Appears on | Means |
|---|---|---|
| `numSlices` | `parallelize`, `range` | **Exact** partition count for driver-side collections |
| `minPartitions` | `textFile`, `wholeTextFiles`, `hadoopFile`, … | **Lower bound** for file reads; Spark may give more if files are large |

- Mnemonic: **driver collection → exact (`numSlices`); file read → at-least (`minPartitions`)**.
- **Default when omitted:** both fall back to `sc.defaultParallelism` (driven by `spark.default.parallelism` config — usually total cluster cores). So `sc.parallelize([1,2,3])` on a 12-core box creates 12 partitions for 3 elements — most empty. That's why "I see so many empty partitions" happens.

**Bucket 2 — The `path` parameter is more flexible than it looks**

Applies to `textFile`, `wholeTextFiles`, `binaryFiles`, all Hadoop readers. A single string accepts:

- A file: `"data/file.txt"`
- A directory (reads every file in it): `"data/"`
- A glob: `"data/*.csv"`, `"data/2024-{01,02}-*.json"`
- Comma-separated paths: `"data/a.txt,data/b.txt,other/c.txt"`
- Any supported URI scheme: `file://`, `hdfs://`, `s3a://`, `gs://`, `abfss://`, …

Skip the `glob.glob()` + Python `for` loop — the path parameter already does this.

**Bucket 3 — Decoding hazard on text readers**

| Option | Appears on | Means |
|---|---|---|
| `use_unicode` (default `True`) | `textFile`, `wholeTextFiles` | Decode bytes as UTF-8. Non-UTF-8 input (Latin-1, Shift-JIS) throws `UnicodeDecodeError` on the executor — surfaces as a cryptic task failure, not a parse error. Set `use_unicode=False` to get raw `bytes` and decode yourself. |

**Bucket 4 — Optimizer hints (assertions — wrong values corrupt silently):**

| Option | Appears on | Means |
|---|---|---|
| `preservesPartitioning` | `map`, `mapPartitions`, `flatMap`, `mapValues` | "My function does not change the keys, so a downstream `reduceByKey` / `join` can skip a shuffle." Only meaningful on a **pair RDD** (already partitioned by key). Lying here corrupts joins/aggregations with no error. |

- Mnemonic: **only set `preservesPartitioning=True` if your function literally cannot change the key** (e.g. `mapValues`, projection of unrelated fields).

**Retention rule:** when a new option shows up, ask *which bucket?* before recording it. New parameter on `sc.binaryFiles`? `minPartitions` again — bucket 1. New parameter on a `mapPartitions`-family method? Probably `preservesPartitioning` — bucket 4. Most "new" options are members of buckets you already know.

**Deliberately skipped** (real but rarely worth memorising): `recordLength` on `binaryRecords`, `keyClass` / `valueClass` / converters on `sequenceFile` / `hadoopFile`, `batchSize` on Hadoop readers. Look them up if you actually touch those APIs.


### 7 — RDD transformations

Categorise by **whether they shuffle**. Narrow chains pipeline inside one stage; wide ones break the stage and move data across the network. (Concept is in `02-rdds.ipynb`; this is the cheat-sheet.)

**Narrow (no shuffle, same stage):**

| Transformation | Notes |
|---|---|
| `map(f)`, `flatMap(f)`, `filter(f)` | The element-wise basics |
| `mapPartitions(f)`, `mapPartitionsWithIndex(f)` | One call per partition (not per row) — amortise per-row setup like DB connections, model loading |
| `glom()` | Each partition → one list. Useful for inspecting partition contents |
| `sample(withReplacement, fraction)` | Per-partition sampling |
| `pipe(command)` | Forks an external process per partition; pipes records through stdin/stdout |
| `union(other)` | Just concatenates partition lists — narrow despite involving two RDDs |
| `zip(other)`, `zipPartitions(...)` | Narrow, but require matching partition counts/sizes |
| `coalesce(n, shuffle=False)` | Narrow **only when shrinking**. Increasing partitions forces `shuffle=True` |
| `mapValues(f)`, `flatMapValues(f)`, `keys()`, `values()` | Pair-RDD narrow ops. `mapValues` / `flatMapValues` **preserve partitioning** — downstream pair ops on the same key can skip a shuffle |

**Wide (shuffle, new stage):**

| Transformation | Notes |
|---|---|
| `distinct()` | Shuffles to dedupe |
| `groupByKey()` | Ships every value across the network — avoid in favour of `reduceByKey` / `aggregateByKey` when computing an aggregate |
| `reduceByKey(f)`, `aggregateByKey(...)`, `foldByKey(...)`, `combineByKey(...)` | Map-side combine before shuffle — preferred aggregation forms |
| `sortByKey()`, `sortBy(keyfn)` | Range-partitions then sorts — full shuffle |
| `join`, `leftOuterJoin`, `rightOuterJoin`, `fullOuterJoin`, `cogroup` / `groupWith` | Co-locate matching keys → shuffle on both sides |
| `intersection(other)`, `subtract(other)`, `subtractByKey(other)` | Set ops require co-located keys |
| `repartition(n)` | Full shuffle to exact count |
| `partitionBy(n, partitionFunc)` | Hash/range partition by a custom function |
| `cartesian(other)` | Every pair — N×M blow-up, the most expensive wide op |
| `coalesce(n, shuffle=True)` | The wide form (or increasing partitions) |

**Hidden-cost gotchas in the "narrow" column:**

- `zipWithIndex()` is listed as narrow but **triggers a job at definition time** to count each partition's size (so it knows the index offsets). Surprising on a pipeline you thought was still lazy.
- `zipWithUniqueId()` avoids that pre-pass by encoding the partition index into the ID — IDs are unique but not contiguous (same gotcha as `monotonically_increasing_id` in Note 4 territory).

**Two-line rule of thumb:**

- *Aggregating?* Prefer `reduceByKey` / `aggregateByKey` over `groupByKey + map`.
- *Adding columns / transforming values on a pair RDD?* Use `mapValues` / `flatMapValues` to keep the partitioner intact and save the next shuffle.


### 8 — RDD actions

Categorise by **what comes back to the driver**, not by name. The bucket tells you whether the action is safe at scale.

| Action | Returns to driver | Safe at scale? |
|---|---|---|
| `count`, `sum`, `max`, `min`, `mean`, `stdev`, `variance`, `reduce`, `fold`, `aggregate` | One scalar | ✅ |
| `take(n)`, `first()`, `top(n)`, `takeOrdered(n)`, `takeSample(...)` | At most `n` rows | ✅ |
| `countByKey()`, `countByValue()`, `collectAsMap()` | Dict, one entry per distinct key/value | ⚠️ Safe if cardinality is bounded, danger if not |
| `isEmpty()`, `histogram(...)` | Boolean / small tuple | ✅ |
| `collect()`, `toLocalIterator()` | All rows | ❌ — driver OOMs |
| `foreach(f)`, `foreachPartition(f)` | Nothing (side-effect on executors) | ✅ (no driver pressure; watch executor log volume) |
| `saveAsTextFile`, `saveAsObjectFile`, `saveAsSequenceFile`, … | Nothing (writes to storage) | ✅ |

Scalar/aggregate actions like `count()` do **not** ship rows to the driver — each partition emits its partial result; the driver combines `O(P)` numbers. They still iterate every record on executors, though, so `count()` re-runs the full lineage just to produce one integer.

**Beyond the common ones, worth knowing about:**

- **Approximate variants** — `countApprox(timeout, confidence)`, `countApproxDistinct(relativeSD)` (HyperLogLog). On huge data when "roughly N within ε" beats waiting hours for an exact count.
- **Tree-shaped aggregations** — `treeReduce(f, depth)`, `treeAggregate(zero, seqOp, combOp, depth)`. Use when you have thousands of partitions and a single-level reduce would overwhelm the driver. Same return shape as `reduce` / `aggregate`, just a different shuffle topology.
- **Stats convenience** — `stats()` returns a `StatCounter` with `count + sum + mean + min + max + variance` in one pass. Cheaper than five separate scalar actions.
- **Pair-RDD specific** — `lookup(key)` fetches values for a key without `collect`; `collectAsMap()` returns the full RDD as a Python dict (same OOM risk as `collect`, plus duplicate keys collapse silently).
- **Save-to-storage family beyond the three above** — `saveAsHadoopFile`, `saveAsNewAPIHadoopFile`, `saveAsHadoopDataset`, `saveAsNewAPIHadoopDataset`, `saveAsPickleFile`. All in the "nothing to driver, just writes" bucket.

**RDD actions ≠ DataFrame actions** (concept is identical, method names diverge):

- RDDs expose numeric aggregates as direct actions: `rdd.mean()`, `rdd.reduce(...)`, `rdd.fold(...)`.
- DataFrames put those behind a transformation + action combo: `df.agg(F.mean("x")).first()[0]`. `.agg()` is a transformation returning a one-row DataFrame; `.first()` / `.collect()` triggers it.
- DataFrame-only verbs: `show()`, `toPandas()`, `printSchema()`, `summary()`, `describe()`, `write.<format>(...)` (replaces the RDD `saveAsX` family).
- RDD-only verbs: `reduce`, `fold`, `aggregate`, `top`, `takeOrdered`, `takeSample`, `countByValue`, `lookup`, `collectAsMap`, `histogram`, `treeReduce`, `treeAggregate`, all the `saveAsX` writers.


### 9 — The one pipeline that anchors Notes 7 & 8

**Scenario** (replay this in your head — it's the spine):

> Two years of raw bank card transactions in object storage. Find customers whose total monthly spend grew more than 20% month-over-month. Enrich with the customer dimension. Cross-check against a known-fraud set. Write the result. Log diagnostics along the way.

Each step below names the op, then says **why that one and not the neighbour** — the contrast is what makes it stick.

---

**1. Ingest two years of CSV.**
```python
t24 = sc.textFile("s3a://.../2024/*.csv", minPartitions=200)
t25 = sc.textFile("s3a://.../2025/*.csv", minPartitions=200)
raw = t24.union(t25)
```
- `textFile` *(not `wholeTextFiles`)* — line-oriented data.
- `union` *(not `+`)* — RDDs aren't lists; `union` is the narrow concat of two partition sets.
- `minPartitions=200` *(not default)* — `defaultParallelism` might give too few partitions on a fat file.

**2. Parse with per-partition setup + diagnostic counter.**
```python
parse_failures = sc.accumulator(0)
parsed = raw.mapPartitions(parse_partition)   # compiles regex once per partition
```
- `mapPartitions` *(not `map`)* — amortise the regex compile.
- `parse_partition` uses `yield` (zero/one/many records per line) — implicit `flatMap` for bad lines and multi-record lines.
- `parse_failures.add(1)` inside on parse errors → diagnostics without breaking laziness.

**3. Drop non-APPROVED.**
```python
txns = parsed.filter(lambda t: t.status == "APPROVED")
```
Pure narrow. `filter` is the obvious one.

**4. Cache** — we're about to use `txns` twice (distinct count + aggregation).
```python
txns.cache()
```
Without this, the lineage from step 1 replays twice. Note 7's wide-vs-narrow buckets matter; Note 2 (lazy re-execution) bites here.

**5. Sanity log: distinct customers.**
```python
n_customers = txns.map(lambda t: t.customer_id).distinct().count()
```
- `distinct` shuffles to dedupe.
- **Aside:** on a billion-row RDD where exact count isn't needed, swap for `countApproxDistinct(relativeSD=0.01)` — HyperLogLog, no shuffle.

**6. Per-(customer, month, category) spend.**
```python
by_cat = (txns
    .map(lambda t: ((t.customer_id, t.year_month, t.category), t.amount))
    .reduceByKey(lambda a, b: a + b))
```
- `reduceByKey` *(not `groupByKey + sum`)* — map-side combine, far less shuffle traffic.
- `reduceByKey` *(not `aggregateByKey`)* — value type == reducer type (both `Double`).
- **Aside:** for *average* not sum, switch to `aggregateByKey((0.0, 0))` carrying `(sum, count)`, then divide at the end. That's the canonical "reducer type ≠ value type" trigger.

**7. Roll up: drop the category dimension.**
```python
by_month = (by_cat
    .map(lambda kv: ((kv[0][0], kv[0][1]), kv[1]))   # ((cid, ym), amount)
    .reduceByKey(lambda a, b: a + b))
```
- Plain `map` then `reduceByKey` — we *change the key*, so `mapValues` doesn't fit (`mapValues` is only when keys are untouched).
- Two `reduceByKey`s in a row are fine — the second has fewer keys than the first, so it shuffles less.

**8. Enrich with customer dimension (small → broadcast).**
```python
b_cust = sc.broadcast(customer_dict)                  # ~10k entries, tiny
keyed = by_month.map(lambda kv: (kv[0][0], (kv[0][1], kv[1])))
enriched = keyed.mapValues(lambda v: (v, b_cust.value[v[0]]))   # no shuffle
```
- Broadcast + `mapValues` *(not `join(customer_rdd)`)* — for a small dimension, broadcast avoids the shuffle a `join` would pay on both sides.
- `mapValues` *(not `map`)* — preserves the partitioner, so a later wide op on `customer_id` stays cheap.
- **Aside:** `join` is correct when both sides are large. `cogroup` is the rare relative — use it only when you need *all values from both sides per key* in one structure.

**9. MoM growth per customer.**
```python
growth = (enriched
    .map(lambda kv: (kv[0], kv[1][0]))           # (cid, (ym, total))
    .groupByKey()                                  # all months per customer
    .mapValues(compute_mom_deltas))                # sort months, emit growth
```
- `groupByKey` *is* the wide op we usually avoid — but here we genuinely need all values per key, and the per-customer fan-out is small (≤ 24 months). The map-side-combine forms (`reduceByKey` etc.) don't help if the function isn't associative-summable.
- `mapValues` *(not `map`)* — same reasoning as step 8.
- **This is where RDDs hurt and DataFrames win.** Real code: switch to `lag(total) OVER (PARTITION BY customer_id ORDER BY year_month)` in Spark SQL. The fact that you keep wanting window functions is *exactly* why DataFrame APIs displaced RDDs for analytics.

**10. Filter to >20% growth, flatten.**
```python
hot = growth.flatMapValues(lambda deltas: [d for d in deltas if d.pct > 0.20])
```
- `flatMapValues` *(not `flatMap` then re-key)* — one pass, preserves partitioner.

**11. Headline numbers.**
```python
top10  = hot.top(10, key=lambda kv: kv[1].pct)
distro = hot.values().map(lambda d: d.pct).stats()
```
- `top` *(not `sortBy(...).take(10)`)* — `top` is one shuffle; `sortBy + take` is a full sort.
- `stats()` *(not five separate `count/sum/mean/min/max` actions)* — one pass returns a `StatCounter` with all of them.

**12. Cross-check against fraud set.**
```python
suspicious = hot.keys().distinct().subtract(fraud_set)
```
- `subtract` *(not `filter(lambda c: c not in fraud_collected)`)* — distributed; the alternative `collect()`s the fraud set to the driver.
- **Aside:** `intersection` for the overlap; `subtractByKey` for pair RDDs where only the key matters.

**13. Write.**
```python
hot.coalesce(8).saveAsTextFile(out)
# DataFrame world: hot_df.coalesce(8).write.format("delta").save(out)
```
- `coalesce` *(not `repartition`)* — shrinking 200 → 8 without a shuffle.
- `saveAsTextFile` *(not `collect()` + driver-side write)* — distributed, no driver OOM.

**14. Wrap up.**
```python
print("rows written :", hot.count())
print("parse fails  :", parse_failures.value)
txns.unpersist()
```
- `count` is the safe scalar action (Note 8).
- Accumulators give diagnostics across the run (Note 7's "writes from executors, reads on driver" model).
- Always `unpersist()` what you `cache()` — it holds executor memory.

---

**Operations this pipeline does NOT use** (the rare-op mnemonic list — one line each, this is what gives you completeness without bloating the spine):

- `cartesian` — every pair; N×M; never without thinking twice.
- `treeReduce` / `treeAggregate` — reduce when you have 10,000+ partitions and a flat reduce would crush the driver.
- `partitionBy(n, partitionFunc)` — partition once by key, save many shuffles in repeated joins.
- `zipWithIndex` — row numbers; secretly triggers a job at definition time.
- `zipWithUniqueId` — IDs without the pre-pass; not contiguous.
- `zip(other)` / `zipPartitions` — element-wise pair-up of two RDDs; requires matching partition counts.
- `pipe(cmd)` — fork an external process per partition; pipes stdin/stdout.
- `glom` — each partition → one list; debugging only.
- `sample(replace, frac)` / `takeSample(replace, n)` — fast random subset for prototyping.
- `lookup(key)` — values for a key without `collect`; uses the partitioner if present.
- `collectAsMap` — collect into a dict; same OOM risk as `collect`, duplicates collapse silently.
- `histogram(buckets)` — numeric distribution by bucket; one pass.
- `countByKey` / `countByValue` — small-cardinality categorical frequencies.
- `isEmpty` — fast emptiness check; just probes the first partition.
- `foldByKey` / `combineByKey` — neighbours of `reduceByKey` / `aggregateByKey`; rarely the right reach.
- `first()` — same as `take(1)[0]`.
- `takeOrdered(n)` — `top(n)` reversed.

---

**Retrieval drill** (this is what actually cements memory — the table doesn't):

Once a week, *before* opening this notebook, sit down and replay the pipeline from step 1 to step 14. Name the op at each step out loud (or write it). Whatever you stumble on is the gap to study — not the rest of the table. Re-reading feels productive but does almost nothing; *trying to recall and failing* is what burns the connection in.

After a month, you should be able to walk steps 1–14 cold. After two months, you should be able to walk the rare-ops list too.


    ---

_New entries go below. Template:_

```
### N — title

One- or two-sentence observation. Code only if it adds something the practice notebooks don't already show.
```
